# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PaNavar369/Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row represents one page-level search/content observation for a defined time period. I will use March 2026 as the development window. The analysis will focus on observable ranking and content signals that are available at the decision moment, with page movement/visibility used as the outcome or ranking target.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Potential candidates from your existing lane:

1)content_age_days

2)ctr

3)avg_position

4) word_count

5)impressions_90d or another historical visibility measure

But don't automatically use all five. We need to check the actual warehouse fields and determine whether each is available before the decision moment.

Label:

trend direction.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
%pip -q install duckdb huggingface_hub pandas

In [5]:
import os
import getpass
import duckdb
import pandas as pd

# Get Hugging Face token from Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

# Connect DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [9]:
# Show the actual columns in the warehouse table

schema = con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [10]:
# Query 1 — Grain verification
# Check whether client + content + report date uniquely identifies a row

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {TABLES["fact_daily"]}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
"""

result = con.sql(query).df()

print(f"Duplicate client/content/date combinations: {len(result):,}")
display(result.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client/content/date combinations: 0


,client_hash_id,content_hash_id,report_date,row_count


One row represents one content/page for one client on one report date. I will use March 2026 as the development window. The grain check found 0 duplicate client/content/date combinations in the March 2026 slice.

In [11]:
# Query 2 — March 2026 row count and date range

query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES["fact_daily"]}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

result = con.sql(query).df()

display(result)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


March 2026 contains 9,841,378 observations, spanning from March 1, 2026 to March 31, 2026.

In [12]:
# Query 3 — GSC availability

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM {TABLES["fact_daily"]}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

result = con.sql(query).df()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


GSC availability: Of the 9,841,378 March 2026 observations, 3,611,061 have GSC data available when filtered using gsc_data_available IS TRUE.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Data limitation: This analysis can identify observed and directional relationships between search/content signals and page movement, but it cannot establish causality or prove that a specific signal causes ranking or visibility changes. The March 2026 slice also has incomplete GSC availability: only 3,611,061 of 9,841,378 observations have gsc_data_available IS TRUE. Therefore, conclusions based on GSC signals may not represent the full dataset. The analysis should be treated as decision-support rather than a prediction of Google's ranking system.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.